# ДЗ 7. Срезы orders.csv: groupby, pivot_table, merge
Срок: **до воскресенья 29.11, 24:00**.
Сдача: папка `hw07` репозитория `dsa-homework` (`hw07.ipynb`): `git add hw07` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать. Рядом с каждой долей или средним указывайте, на скольких заказах оно посчитано.

Инструменты: всё с уроков 1–7. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
t = pd.DataFrame({"c": ["a", "b", "a"], "x": [1, 2, 3]})
print(t.groupby("c")["x"].sum().tolist())
# 2
t = pd.DataFrame({"c": ["a", "a", "b"], "d": [1, None, 3]})
print(t.groupby("c")["d"].count().tolist(), t.groupby("c").size().tolist())
# 3
t = pd.DataFrame({"c": ["a", "a", "b"], "m": [9, 9, 10], "x": [10, 30, 5]})
print(t.pivot_table(index="c", columns="m", values="x").loc["a", 9])
# 4
a = pd.DataFrame({"id": [1, 2, 3]})
b = pd.DataFrame({"id": [1, 2], "ch": ["x", "y"]})
print(len(a.merge(b, on="id")), len(a.merge(b, on="id", how="left")))
# 5
b = pd.DataFrame({"id": [1, 1, 2], "ch": ["x", "x", "y"]})
print(len(a.merge(b, on="id", how="left")))
```

Ваши ответы и объяснения:

1. 
2. 
3. 
4. 
5. 

In [1]:
# Загрузка orders.csv и customers.csv: в Colab нажмите «Выбрать файлы» и укажите оба файла из LMS.
# Если файлы уже лежат рядом с ноутбуком, загрузка пропускается.
import os
if not (os.path.exists("orders.csv") and os.path.exists("customers.csv")):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv и customers.csv в одну папку с ноутбуком")
print("orders.csv:", os.path.exists("orders.csv"), "| customers.csv:", os.path.exists("customers.csv"))

orders.csv: True | customers.csv: True


In [2]:
# Очистка урока 6 одной ячейкой: суммы, города, дубли, даты, TEST
import pandas as pd
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

df = pd.read_csv("orders.csv")
df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", ""))
df["city"] = df["city"].str.strip().str.lower().map(CITY)
df = df.drop_duplicates()
df["order_date"] = pd.to_datetime(df["order_date"])
df["month"] = df["order_date"].dt.month
df = df[df["customer_id"] != "TEST"].copy()    # copy: дальше добавим столбцы
df.shape

(5280, 9)

In [3]:
df["cancel"] = df["status"] == "отменён"
paid = df[df["status"] == "оплачен"]
cu = pd.read_csv("customers.csv")
len(paid), cu.shape

(4739, (2200, 4))

Можно вместо ячейки очистки взять свою функцию `clean()` из ДЗ 6. Проверка: 5 280 строк, 4 739 оплаченных.

## [База] 1. Сводка по городам одной таблицей
Через `agg` с именами: число заказов, доля отмен, средний срок доставки по городам. Добавьте столбец «выручка оплаченных»: `by_city["revenue"] = paid.groupby("city")["amount"].sum()` (Pandas выровняет по ярлыкам, урок 5). Отсортируйте по выручке.
Какой город выделяется сразу по двум показателям?

In [4]:
by_city = df.groupby("city").agg(
    orders=("order_id", "count"),
    cancel_rate=("cancel", "mean"),
    avg_days=("delivery_days", "mean"),
)
by_city["revenue"] = paid.groupby("city")["amount"].sum()
by_city = by_city.sort_values("revenue", ascending=False)
by_city["cancel_rate"] = (by_city["cancel_rate"] * 100).round(1)
by_city["avg_days"] = by_city["avg_days"].round(2)
by_city

,orders,cancel_rate,avg_days,revenue
city,,,,
Алматы,2116,7.8,2.00,29144090.0
Астана,1339,8.8,2.02,17884750.0
Шымкент,776,20.6,4.45,8602660.0
Караганда,519,9.1,1.94,7020200.0
Актобе,387,10.6,2.07,5117380.0
Атырау,143,7.7,2.04,1893850.0


вывод для менеджера: по выручке шымкент на третьем месте, но он выделяется тем, что отменяют там 20.6% и в среднем доставка идёт 4.45 дня против ~2 дня в других городах. То есть в шыме доставка в два раза дольше и отмен в два раза больше. Возможно это проблема в шымкенте

## [База] 2. ДЗ 4 одной строкой
В ДЗ 4 вы считали категории циклами и NumPy. Повторите через `pivot_table`: выручка оплаченных по категориям и месяцам, изменение в %, средний чек по категориям и месяцам.
Сверьте с ответами ДЗ 4: лидер Электроника 34,4 млн ₸, Одежда −7,2%, средний чек Электроники в октябре 33 600 ₸. Сколько строк кода было и сколько стало?

In [5]:
rev = paid.pivot_table(index="category", columns="month", values="amount", aggfunc="sum")
rev["chg_%"] = ((rev[10] / rev[9] - 1) * 100).round(1)
rev["total"] = rev[9] + rev[10]
rev.sort_values("total", ascending=False)

month,9,10,chg_%,total
category,,,,
Электроника,17402330.0,16968100.0,-2.5,34370430.0
Одежда,9167100.0,8505730.0,-7.2,17672830.0
Дом,4997730.0,5125790.0,2.6,10123520.0
Красота,2610950.0,2678990.0,2.6,5289940.0
Книги,1065300.0,1140910.0,7.1,2206210.0


In [6]:
avg = paid.pivot_table(index="category", columns="month", values="amount", aggfunc="mean").round(0)
n = paid.pivot_table(index="category", columns="month", values="amount", aggfunc="count")
avg.join(n, lsuffix="_avg", rsuffix="_orders")

month,9_avg,10_avg,9_orders,10_orders
category,,,,
Дом,10499.0,10768.0,476,476
Книги,4020.0,3816.0,265,299
Красота,6187.0,6159.0,422,435
Одежда,13521.0,13046.0,678,652
Электроника,32773.0,33600.0,531,505


вывод для менеджера: топ 1 электроника 34 млн тг за 2 месяца, сильнее всех упала Одежда на −7.2%. средний чек Электроники в октябре 33600тг, электроника приносит половину выручки, а одежда сильнее других теряет деньги

## [База] 3. Отмены и срок доставки
По `delivery_days`: число заказов и доля отмен. Во сколько раз отменяют чаще при доставке за 8 дней, чем за 1 день?
Что это значит для Шымкента, где в октябре доставка 5–8 дней?

In [7]:
by_days = df.groupby("delivery_days").agg(
    orders=("order_id", "count"),
    cancel_rate=("cancel", "mean"),
)
ratio = by_days.loc[8, "cancel_rate"] / by_days.loc[1, "cancel_rate"]
by_days["cancel_rate"] = (by_days["cancel_rate"] * 100).round(1)
print(by_days)
print()
print(f"8 дней / 1 день: в {ratio:.1f} раза")

               orders  cancel_rate
delivery_days                     
1.0              1557          4.2
2.0              1570          8.0
3.0              1590         13.4
5.0                86         22.1
6.0               120         21.7
7.0                95         34.7
8.0               104         43.3

8 дней / 1 день: в 10.2 раза


In [8]:
df[df["city"] == "Шымкент"].groupby("month")["delivery_days"].agg(["count", "mean", "min", "max"])

,count,mean,min,max
month,,,,
9,349,2.031519,1.0,3.0
10,405,6.535802,5.0,8.0


вывод для менеджера: чем дольше доставка, тем чаще отменяют, при доставке за 1 день отменяют 4.2% заказов, за 3 дня уже 13.4%, а за 8 дней 43.3%. В Шымкенте в сентябре доставляли за 1-3 дня, а в октябре только за 5-8 дней. пока доставку в щымкенте не починят отмены будут расти

## [Стандарт] 4. Исправьте код от AI
AI посчитал изменение выручки по городам и пишет менеджеру: «В Шымкенте выручка выросла на 2,4%, проблем нет». В коде **две** ошибки. Найдите их, исправьте и напишите менеджеру правильную цифру.
```python
p = df.pivot_table(index="city", columns="month", values="amount")
p["chg"] = p[10] / p[9] - 1
print(p["chg"].round(3))
```

In [9]:
p = df.pivot_table(index="city", columns="month", values="amount")
p["chg"] = p[10] / p[9] - 1
print(p["chg"].round(3))

city
Актобе      -0.046
Алматы      -0.005
Астана      -0.028
Атырау      -0.092
Караганда    0.064
Шымкент      0.024
Name: chg, dtype: float64


In [10]:
p = paid.pivot_table(index="city", columns="month", values="amount", aggfunc="sum")
p["chg"] = p[10] / p[9] - 1
print((p["chg"] * 100).round(1))

city
Актобе       -1.4
Алматы        1.0
Астана       -2.0
Атырау        5.0
Караганда    -7.6
Шымкент     -11.3
Name: chg, dtype: float64


1) У pivot_table по умолчанию aggfunc="mean", поэтому код считает не прибыль, а средний чек. В Щымкенте средний чек действительно вырос на 2.4%, но выручка считается как сумма, нужно aggfunc="sum"

2) Взята вся таблица df, вместе с отменёнными заказами, прибыль считают только по оплаченным, поэтому нужна таблица paid

выручка в Шымкенте в октябре упала на 11.3% (с 4,56 млн до  4,04 млн тг). Это самое сильное падение среди всех городов, так что проблема есть, и серьёзная. Вывод "выросла на 2.4%, проблем нет" неправильный

## [Стандарт] 5. Возраст клиентов
Соедините заказы с клиентами (`merge`, `how="left"`, проверьте `len`). Возрастная группа: `m["age_group"] = m["age"] // 10 * 10`.
По группам: число заказов, доля отмен, средний и медианный чек оплаченных.
AI предлагает: «Группа 60 покупает дороже всех и чаще отменяет, сделаем для неё отдельную акцию». Какие возрасты попали в группу 60 и сколько в ней заказов? Подтверждает ли медиана вывод AI?

In [11]:
m = df.merge(cu, on="customer_id", how="left")
print(len(df), len(m), "пропусков в age:", m["age"].isna().sum())

m["age_group"] = m["age"] // 10 * 10
mp = m[m["status"] == "оплачен"]

ages = m.groupby("age_group").agg(orders=("order_id", "count"), cancel_rate=("cancel", "mean"))
ages["cancel_rate"] = (ages["cancel_rate"] * 100).round(1)
ages["paid_orders"] = mp.groupby("age_group").size()
ages["mean_check"] = mp.groupby("age_group")["amount"].mean().round(0)
ages["median_check"] = mp.groupby("age_group")["amount"].median()
ages

5280 5280 пропусков в age: 0


,orders,cancel_rate,paid_orders,mean_check,median_check
age_group,,,,,
10,229,8.7,209,14045.0,10700.0
20,1226,9.6,1108,14130.0,10100.0
30,1241,11.1,1103,15142.0,10670.0
40,1170,11.1,1040,14793.0,10050.0
50,1294,9.2,1175,14672.0,10240.0
60,120,13.3,104,16779.0,10495.0


In [12]:
print("возрасты в группе 60:", sorted(m.loc[m["age_group"] == 60, "age"].unique().tolist()))
print("возрасты в группе 10:", sorted(m.loc[m["age_group"] == 10, "age"].unique().tolist()))
print("заказов в группе 60:", (m["age_group"] == 60).sum())

возрасты в группе 60: [60]
возрасты в группе 10: [18, 19]
заказов в группе 60: 120



Вывод для менеджера: В группу 60 попали только клиенты ровно 60 лет, потому что старше 60 в базе никого нет. В группе всего 120 заказов. Средний чек у неё действительно самый высокий 16779 тг, но медиана 10495тг. Значит, высокий средний чек дают несколько дорогих покупок, а обычный покупатель 60 лет тратит столько же, сколько остальные. Доля отмен 13.3% означает 16 отмен из 120 заказов, при такой маленькой выборке это может быть случайностью. Медиана вывод ии не подтверждает, отдельная акция для группы 60 не обоснована

## [Стандарт] 6. Повторные покупки
Сколько заказов у каждого клиента (`groupby("customer_id").size()`)? Сколько клиентов сделали 1, 2, 3… заказа (`value_counts().sort_index()`)? Какая доля клиентов купила больше одного раза?
Сколько клиентов заказывали и в сентябре, и в октябре (`groupby("customer_id")["month"].nunique()`)? Сколько клиентов из customers.csv не заказали ничего?

In [13]:
per_client = df.groupby("customer_id").size()
dist = per_client.value_counts().sort_index()
print(dist)
print()
print("клиентов с заказами:", len(per_client))
print(f"купили больше одного раза: {(per_client > 1).sum()} ({(per_client > 1).mean() * 100:.1f}%)")

1     501
2     563
3     434
4     293
5     127
6      58
7      15
8       9
9       1
10      1
Name: count, dtype: int64

клиентов с заказами: 2002
купили больше одного раза: 1501 (75.0%)


In [14]:
months_per_client = df.groupby("customer_id")["month"].nunique()
both = (months_per_client == 2).sum()
print(f"заказывали и в сентябре, и в октябре: {both} из {len(months_per_client)} ({both / len(months_per_client) * 100:.1f}%)")

never = ~cu["customer_id"].isin(df["customer_id"])
print(f"в customers.csv {len(cu)} клиентов, ничего не заказали: {never.sum()}")

заказывали и в сентябре, и в октябре: 1064 из 2002 (53.1%)
в customers.csv 2200 клиентов, ничего не заказали: 198


вывод для менеджера: заказывали 2 002 клиента. Один заказ у 501 клиента, два у 563, три у 434,  10 заказов у одного клиента. Больше одного раза купили 1 501 клиент, то есть 75,0%: три из четырёх покупателей возвращаются. В оба месяца заказывали 1 064 клиента, так что у магазина хорошее ядро постоянных покупателей. Ещё 198 клиентов из базы зарегистрировались, но не сделали ни одного заказа

## [Челлендж] 7. Сколько стоил сбой доставки в Шымкенте
Допустим, в октябре в Шымкенте было бы столько же заказов и тот же средний чек оплаченных, а доля отмен осталась как в сентябре. Какой была бы выручка Шымкента в октябре? Сколько компания потеряла из-за отмен?
Сравните потерю с падением всей компании (823 890 ₸). Что было бы с выручкой компании без сбоя?

In [15]:
# 7

## [Челлендж] 8. Органика −6,0%: сигнал или шум?
Разбейте клиентов из customers.csv **случайно** на три группы в тех же пропорциях, что и каналы (`np.random.default_rng(7)`, `rng.choice`, `map`). Посчитайте изменение выручки оплаченных окт/сент для каждой группы. Повторите 100 раз в цикле.
В каком диапазоне меняется выручка случайной группы размером с органику (5% и 95% перцентили)? Можно ли по «органика −6,0%» делать вывод о SEO?

In [16]:
# 8